In [1]:
import io
import zlib

import torch
import numpy as np
from sklearn.metrics import r2_score

import data.codecs as codecs

In [2]:
from importlib import reload
reload(codecs)

<module 'data.codecs' from '/data/connor/fmri-fm/src/data/codecs.py'>

In [3]:
torch.manual_seed(42)

In [4]:
P = 10
D = 1024
N = 64
d = 16
max_bins = 256
parc = torch.randint(0, P + 1, (D,))

In [5]:
codec = codecs.ParcelPCAQuantize.from_parcellation(parc, n_components=d, max_bins=max_bins)

In [6]:
scale = torch.logspace(1, 0, d)
u = torch.randn(N, P, d) * scale
mask = codec.parcellate.parc_ids >= 0

z = codec.pca.inverse(u, mask)
x = codec.parcellate.inverse(z)

In [7]:
b0 = codecs.encode_tensor(x)

In [8]:
codec.fit(x)

ParcelPCAQuantize(
  (parcellate): Parcellate()
  (pca): PatchPCA()
  (quantize): Quantize()
)

In [9]:
b = codecs.encode_tensor(codec.forward(x))
x_ = codec.inverse(codecs.decode_tensor(b))
print(r2_score(x.numpy(), x_.numpy()))
print(len(b0) / len(b))

0.999550461769104
21.873192541856927


In [10]:
codec2 = codecs.Quantize(D, max_bins=max_bins)
codec2.fit(x)

Quantize()

In [11]:
b2 = codecs.encode_tensor(codec2.forward(x))
x2_ = codec2.inverse(codecs.decode_tensor(b2))
print(r2_score(x.numpy(), x2_.numpy()))
print(len(b0) / len(b2))

0.9994768500328064
3.5997589003350345


In [12]:
print(len(b2) / len(b))

6.076293759512938
